# Phase 5 — Generation eval: arms A/B/C on Qwen2.5-3B (base + LoRA)

Runs `eval/eval_mcq.py` and `eval/eval_generation.py` for the three arms that need a
GPU (this dev sandbox is CPU-only, 3.7GB RAM — same constraint Phase 4 hit):

| Arm | What it isolates |
|---|---|
| A. Qwen2.5-3B base, no RAG | Closed-book floor |
| B. Qwen2.5-3B base + RAG | Value of retrieval alone |
| C. Qwen2.5-3B LoRA + RAG | Value of fine-tuning — the deliverable |

Arm D (Claude + RAG) needs no GPU and has already been run locally — its results are
already in `eval/data/mcq_results.json` / `eval/data/generation_results.json`. Both
scripts merge new arms into whatever results file already exists (same pattern as
`eval_retrieval.py`), so upload those two JSON files alongside the input data below and
the file you download at the end will have all four arms in one place.

**Before running:** pick the winning LoRA rank from Phase 4's loss curves
(`notebooks/03_finetune_output.ipynb` — lowest val loss without a clear overfitting
uptick) and set `ADAPTER_RANK` below.

**Files to upload** (all small — this notebook deliberately avoids re-uploading the
corpus or re-running retrieval infrastructure beyond what these two scripts need):
into `/content/drive/MyDrive/derma_force_eval/`:
```
eval/
  eval_mcq.py  eval_generation.py  eval_retrieval.py  build_index.py  judge.py
  data/
    exam_parsed/all.jsonl
    exam_heldout_translations.json
    test.jsonl
    ood_questions.json
    indices/400_80__bge.faiss  400_80__bge.meta.jsonl
    mcq_results.json            <- arm D's results, if already run locally
    generation_results.json     <- ditto
pipeline/data/chunks/chunks.jsonl
backend/
  generators.py  verifier.py  config.py  .env
```
The layout under `/content/derma_force/` mirrors the repo root exactly so the scripts'
relative-path assumptions (`EVAL_DIR.parent / "backend"`, etc.) just work unmodified.

## 1. Setup

In [2]:
!pip install -q -U "transformers>=4.44,<5" "accelerate>=0.33" "peft>=0.12" "bitsandbytes>=0.43" \
    faiss-cpu sentence-transformers rouge-score sacrebleu anthropic python-dotenv

# See 03_finetune.ipynb's identical warning: Colab pre-imports an older transformers
# before this cell runs; if the version assert below fails, Runtime > Restart session
# then Runtime > Run all again (re-running cells in place will NOT pick up the install).

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 144.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 16.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have h

In [3]:
import torch
import transformers

assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0))

_tf_version = tuple(int(x) for x in transformers.__version__.split(".")[:2])
assert _tf_version >= (4, 44), (
    f"transformers still {transformers.__version__} — Runtime > Restart session, then Run all again."
)

NVIDIA A100-SXM4-40GB


In [4]:
from google.colab import drive
drive.mount('/content/drive')

EVAL_DRIVE_DIR = '/content/drive/MyDrive/derma_force_eval'   # see the file layout above
ADAPTER_DIR = '/content/drive/MyDrive/derma_force_qlora'      # Phase 4's saved adapters

# Recreate the repo layout under /content/derma_force/ by symlinking from Drive, so
# eval_mcq.py / eval_generation.py's relative paths (EVAL_DIR.parent / "backend", etc.)
# resolve exactly as they do when run from the real repo.
import os
REPO_ROOT = '/content/derma_force'
if not os.path.exists(REPO_ROOT):
    os.symlink(EVAL_DRIVE_DIR, REPO_ROOT)
print("repo root:", REPO_ROOT, "->", os.readlink(REPO_ROOT) if os.path.islink(REPO_ROOT) else "(real dir)")
!ls {REPO_ROOT}

Mounted at /content/drive
repo root: /content/derma_force -> /content/drive/MyDrive/derma_force_eval
backend		data  eval_generation.py  eval_retrieval.py  pipeline
build_index.py	eval  eval_mcq.py	  judge.py


## 2. Pick the winning LoRA rank

From Phase 4's loss curves: val loss for all three ranks plateaus flat from step ~50
onward while train loss keeps dropping fastest for r=32 — the widening train/val gap
scaling with rank is the overfitting signal. Absent a clear winner on val loss alone,
the smallest rank that reaches the plateau (least overfit, cheapest) is the defensible
default; override `ADAPTER_RANK` after actually reading the curves.

In [5]:
ADAPTER_RANK = 16  # <-- set from notebooks/03_finetune_output.ipynb's loss_curves.png
ADAPTER_PATH = f"{ADAPTER_DIR}/r{ADAPTER_RANK}/adapter"
assert os.path.isdir(ADAPTER_PATH), f"no adapter at {ADAPTER_PATH} — check ADAPTER_RANK / Drive path"
print("using adapter:", ADAPTER_PATH)

using adapter: /content/drive/MyDrive/derma_force_qlora/r16/adapter


## 3. MCQ eval (`eval_mcq.py`) — arms A, B, C

In [6]:
%cd {REPO_ROOT}/eval
!python3 eval_mcq.py --arms A,B --device cuda
!python3 eval_mcq.py --arms C --adapter {ADAPTER_PATH} --device cuda

/content/drive/MyDrive/derma_force_eval/eval
286 held-out questions (2025+2026, text_answerable+image_dependent)
258 usable for MCQ scoring (0 no translation yet, 28 incomplete options)
2026-09-03 05:49:09.616982: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-03 05:49:09.684467: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
modules.json: 100% 349/349 [00:00<00:00, 2.54MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 1.17MB/s]
README.md: 94.8kB [00:00, 124MB/s]
sentence_bert_config.json: 100

## 4. Generation eval (`eval_generation.py`) — synthetic test set + out-of-corpus abstention, arms A, B, C

In [9]:
!python3 eval_generation.py --arms A,B --set both --device cuda
!python3 eval_generation.py --arms C --adapter {ADAPTER_PATH} --set both --device cuda

2026-09-03 08:03:33.545413: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-03 08:03:33.612504: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading checkpoint shards: 100% 2/2 [00:06<00:00,  3.45s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  [A/test] 10/99
  [A/test] 20/99
  [A/test] 30/99
  [A/test] 40/99
  [A/test] 50/99
  [A/test] 60/99
  [A/test] 70/99
  [A/test] 80/99
  [A/test] 90/99
  [A/tes

## 5. Pull the merged results back

Both files now have all four arms (A/B/C from this run, D already merged in from the
local run if you uploaded it in step 1). Copy them back to Drive so they survive the
runtime disconnecting, then download to the real repo's `eval/data/`.

In [8]:
import shutil
for fname in ["mcq_results.json", "generation_results.json"]:
    shutil.copy(f"{REPO_ROOT}/eval/data/{fname}", f"{EVAL_DRIVE_DIR}/data/{fname}")
    print("saved ->", f"{EVAL_DRIVE_DIR}/data/{fname}")

saved -> /content/drive/MyDrive/derma_force_eval/data/mcq_results.json
saved -> /content/drive/MyDrive/derma_force_eval/data/generation_results.json


## Next steps (Phase 5, continued)

With `mcq_results.json` / `generation_results.json` holding all four arms:
- Build the accuracy-by-arm-and-bucket table, the ceiling calculation, and the
  position-bias / abstention analyses (`derma_guide_plan.md`'s Phase 5 Analysis list).
- Wire the verifier badge into `frontend/` for the live demo.
- Move into Phase 6: plots, report, slides.